# AI Content Moderation — Model Training Notebook

This notebook fine-tunes a transformer model to classify social media text
as **SAFE**, **OFFENSIVE**, or **HATE**, using the merged Davidson + OLID
dataset prepared by `scripts/preprocess_data.py`.

**How to use this notebook (first time in Colab? read this):**
1. Run each code cell in order, top to bottom, by clicking the ▶️ button
   on the left of the cell (or pressing `Shift+Enter`).
2. Wait for a cell to finish (the ▶️ turns into a spinning circle while
   running, and a number like `[3]` appears when done) before running the
   next one.
3. When you reach the "Upload your data files" cell, a **Choose Files**
   button will appear — select `train.csv`, `val.csv`, `test.csv`, and
   `olid_official_test.csv` from the `data/processed/` folder (all 4 at
   once is fine).

**Before you run anything: turn on the free GPU.**
Menu bar → `Runtime` → `Change runtime type` → under *Hardware
accelerator* choose **T4 GPU** → `Save`. Do this before running Step 1
below, otherwise training will run on CPU and be much slower.

## Step 1 — Confirm the GPU is on

In [ ]:
!nvidia-smi

If you see a table with a GPU name like `Tesla T4`, you're set.
If you instead see `NVIDIA-SMI has failed...`, go back and do
`Runtime > Change runtime type > T4 GPU > Save`, then run this cell again.

## Step 2 — Install the libraries we need

In [ ]:
!pip install -q transformers datasets scikit-learn evaluate accelerate huggingface_hub

## Step 3 — Upload your data files

Run the cell below, then click **Choose Files** and select all four CSVs
from `data/processed/` in the project zip: `train.csv`, `val.csv`,
`test.csv`, `olid_official_test.csv`.

In [ ]:
from google.colab import files
uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))

## Step 4 — Load the data and check it looks right

In [ ]:
import pandas as pd

train_df = pd.read_csv("train.csv")
val_df = pd.read_csv("val.csv")
test_df = pd.read_csv("test.csv")
external_df = pd.read_csv("olid_official_test.csv")

print("train:", train_df.shape, "val:", val_df.shape,
      "test:", test_df.shape, "external:", external_df.shape)
train_df.head()

## Step 5 — Turn the text labels into numbers the model can use

In [ ]:
LABELS = ["SAFE", "OFFENSIVE", "HATE"]
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}

for df in [train_df, val_df, test_df, external_df]:
    df["label_id"] = df["label"].map(label2id)

train_df["label_id"].value_counts()

## Step 6 — Convert to Hugging Face `Dataset` objects and tokenize

We're using `bert-base-multilingual-cased` for the English baseline
(Milestone 1). To move to Milestone 2 (Hindi/Hinglish) later, you'd change
`MODEL_NAME` below to `google/muril-base-cased` and re-run from here with
Hindi/Hinglish data added in.

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

MODEL_NAME = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_hf_dataset(df):
    return Dataset.from_pandas(
        df[["text", "label_id"]].rename(columns={"label_id": "label"}),
        preserve_index=False,
    )

train_ds = to_hf_dataset(train_df)
val_ds = to_hf_dataset(val_df)
test_ds = to_hf_dataset(test_df)
external_ds = to_hf_dataset(external_df)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)

train_ds = train_ds.map(tokenize, batched=True)
val_ds = val_ds.map(tokenize, batched=True)
test_ds = test_ds.map(tokenize, batched=True)
external_ds = external_ds.map(tokenize, batched=True)

## Step 7 — Load the model and set class weights

Your data is imbalanced (`SAFE` 34%, `OFFENSIVE` 59%, `HATE` only 6.6%).
Without correcting for this, the model can get "good" accuracy while
barely ever predicting `HATE`. These weights push the loss function to
pay more attention to the rare class.

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=3, id2label=id2label, label2id=label2id
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Inverse-frequency weights based on SAFE 34.2% / OFFENSIVE 59.2% / HATE 6.6%
class_weights = torch.tensor([1.0, 0.6, 5.2]).to(device)

## Step 8 — A custom Trainer that uses those class weights

In [ ]:
from transformers import Trainer, TrainingArguments
import torch.nn as nn

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fct = nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

## Step 9 — Define the metrics we care about

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1_macro": f1_score(labels, preds, average="macro"),
    }

## Step 10 — Train the model

This is the step that takes the longest (roughly 10-25 minutes on a free
T4 GPU for this dataset size, 3 epochs). Watch `f1_macro` in the printed
table after each epoch — that's your real headline metric, not accuracy.

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    fp16=True,
    logging_steps=50,
    report_to="none",
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
)

trainer.train()

> **If this cell errors on `eval_strategy`:** your Colab's `transformers`
> version is older and expects `evaluation_strategy` instead. Just rename
> that one argument in the cell above and re-run.

## Step 11 — Evaluate properly

We check performance on two different sets:
- `test.csv` — held out from the same pool the model trained on.
- `olid_official_test.csv` — a completely external set the model has
  never seen, used to check it actually generalized rather than
  memorized.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

def evaluate_on(dataset, name):
    preds = trainer.predict(dataset)
    y_pred = np.argmax(preds.predictions, axis=1)
    y_true = preds.label_ids
    print(f"=== {name} ===")
    print(classification_report(y_true, y_pred, target_names=LABELS))
    print("Confusion matrix (rows=actual, cols=predicted):")
    print(confusion_matrix(y_true, y_pred))
    print()

evaluate_on(test_ds, "Held-out test.csv")
evaluate_on(external_ds, "External olid_official_test.csv")

## Step 12 — Save the model

This saves the fine-tuned model + tokenizer to a folder, zips it, and
downloads it to your computer so you can use it in the FastAPI backend
later.

In [ ]:
model.save_pretrained("./moderation-model")
tokenizer.save_pretrained("./moderation-model")

!zip -rq moderation-model.zip moderation-model
print("Saved and zipped. Starting download...")

from google.colab import files
files.download("moderation-model.zip")

## Step 13 (optional but recommended) — Push the model to Hugging Face Hub

This lets your FastAPI backend load the model directly by name at
startup instead of you having to upload a large file to GitHub/Render.

1. Create a free account: https://huggingface.co/join
2. Create an access token (Write role): https://huggingface.co/settings/tokens
3. Run the cell below and paste the token when it asks.

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# Replace "your-username" with your actual Hugging Face username
repo_name = "your-username/indian-content-moderation-v1"

model.push_to_hub(repo_name)
tokenizer.push_to_hub(repo_name)
print(f"Pushed to https://huggingface.co/{repo_name}")

## You've completed Milestone 1 🎉

You now have a fine-tuned, evaluated model. Next: go to
`docs/04_Implementation_Plan.md`, **Phase 3 (Backend)**, to wire this
model into the FastAPI app.

To extend to Hindi/Hinglish (Milestone 2), see `docs/07_AI_Model.md`
section 6 — you'd add HASOC/HateXplain/HingCorpus data, change
`MODEL_NAME` to `google/muril-base-cased` in Step 6 above, and re-run
this same notebook.